# Harmony4D trên Drive → Google Sheet + manifest JSON

Notebook duyệt thư mục `test` của Harmony4D trên Google Drive. Với mỗi camera, nó lấy File ID của GT (`poses3d.zip`), video gốc, tệp PKL và video WHAM. Kết quả được ghi vào một Google Sheet và một tệp JSON `{"test": [...]}`.

**Cách dùng**
1. Mở notebook trên Colab. Sửa ô **Cấu hình** nếu cần.
2. Chọn `Runtime → Run all`. Lần đầu Colab hỏi quyền truy cập tài khoản Google. Hãy chọn tài khoản có quyền **Editor** trên thư mục dataset, vì notebook cần upload `poses3d.zip` vào đó.
3. Có thể chạy lại nhiều lần. `poses3d.zip` đã có thì được dùng lại. Sheet được ghi tiếp xuống dưới. Tệp JSON được ghi đè.

**Cấu trúc thư mục mong đợi**
```
<gốc>/test/<nhóm>/<X>/processed_data/poses3d/*.npy
                     /<X>/exo/camY/harmony4D_X_camY_ffmpeg.mp4        (hoặc trong camY/images/)
                     /<X>/exo/camY/harmony4D_X_camY_wham/  ← tệp .pkl có X và Y trong tên
                                                           ← tệp .mp4 có output_X_camY trong tên
```
- `processed_data` được tìm ở mọi độ sâu, tối đa 3 cấp dưới `test`. Vì vậy có hay không có thư mục `<nhóm>` (ví dụ `03_grappling2`) đều chạy được.
- Y giữ nguyên như trong tên thư mục: `cam07` cho Y = `07`.

**Về `skip`:** `skip = true` khi thiếu `pkl_file_ID` hoặc `z1`. Đây là nghĩa đang dùng trong repo: `true` = bỏ qua sequence này.

## 1. Cấu hình
Sửa các hằng số dưới đây nếu cấu trúc trên Drive khác.

In [ ]:
ROOT_FOLDER_ID = "1UX3jW_pA2FoJRNsXsjwHBUk5QjmSAlTK"  # thư mục Harmony4D trên Drive
TEST_FOLDER_NAMES = ("test", "test_set")  # tên thư mục test (lấy cái tìm thấy đầu tiên)
MAX_DEPTH = 3              # processed_data nằm sâu tối đa bao nhiêu cấp dưới test
FPS = 20                   # giá trị "fps" ghi vào JSON
ZIP_NAME = "poses3d.zip"
SHEET_NAME = "harmony4d_test_index"          # Google Sheet trong My Drive (tạo nếu chưa có)
LOCAL_JSON = "/content/harmony4d_test.json"  # cũng được upload lên My Drive cùng tên
LOCAL_WORK = "/content/h4d_work"             # chỗ tạm để tải poses3d và nén
DOWNLOAD_WORKERS = 8       # số luồng tải song song
TIMEZONE = "Asia/Ho_Chi_Minh"                # múi giờ cho cột time (Colab mặc định UTC)

Thư viện và hằng số nội bộ. Mọi thư viện đều có sẵn trên Colab.

In [ ]:
import io, os, re, json, shutil, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from zoneinfo import ZoneInfo

import google.auth
import gspread
import pandas as pd
from google.colab import auth, files
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from tqdm.auto import tqdm

FOLDER_MIME = "application/vnd.google-apps.folder"
SHORTCUT_MIME = "application/vnd.google-apps.shortcut"
GOOGLE_APPS_PREFIX = "application/vnd.google-apps."  # Docs, Sheets...: không tải được dạng tệp
CAM_RE = re.compile(r"^cam(\d+)$")
SHEET_HEADER = ["STT", "file PKL", "original_video", "wham_video", "GT", "cam", "action", "time"]
_tls = threading.local()  # mỗi luồng tải giữ một kết nối Drive riêng

## 2. Đăng nhập
`auth_services()` đăng nhập Google trong Colab rồi trả về ba đối tượng:
- `creds`: thông tin đăng nhập. Mỗi luồng tải dùng nó để tạo kết nối riêng.
- `drive`: Drive API v3, dùng để liệt kê, tải và upload tệp.
- `gc`: client `gspread`, dùng để ghi Google Sheets.

In [ ]:
def auth_services():
    """Đăng nhập Google trong Colab, trả về (creds, drive, gc)."""
    auth.authenticate_user()
    creds, _ = google.auth.default()
    drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    gc = gspread.authorize(creds)
    return creds, drive, gc

## 3. Tiện ích Drive
`is_folder(item)` cho biết một phần tử Drive (dict có `mimeType`) có phải thư mục không.

In [ ]:
def is_folder(item):
    """True nếu phần tử Drive là thư mục."""
    return item.get("mimeType") == FOLDER_MIME

`list_children(drive, parent_id, name_contains=None)` liệt kê các phần tử con trực tiếp của một thư mục.
- Đọc hết mọi trang kết quả, có hỗ trợ Shared Drive.
- Shortcut được thay bằng tệp hoặc thư mục đích của nó, nên vẫn duyệt được như phần tử thật.
- Nếu có `name_contains`, Drive chỉ trả về phần tử có tên bắt đầu bằng chuỗi đó. Cách này tránh phải liệt kê hàng nghìn ảnh trong `images/`.

In [ ]:
def list_children(drive, parent_id, name_contains=None):
    """Mọi phần tử con trực tiếp của parent_id: [{id, name, mimeType}, ...]."""
    query = f"'{parent_id}' in parents and trashed = false"
    if name_contains:
        safe = name_contains.replace("\\", "\\\\").replace("'", "\\'")
        query += f" and name contains '{safe}'"
    fields = "nextPageToken, files(id, name, mimeType, shortcutDetails)"
    items, token = [], None
    while True:
        resp = drive.files().list(
            q=query, fields=fields, pageSize=1000, pageToken=token, orderBy="name",
            supportsAllDrives=True, includeItemsFromAllDrives=True,
        ).execute(num_retries=3)
        for f in resp.get("files", []):
            if f["mimeType"] == SHORTCUT_MIME:
                target = f.get("shortcutDetails", {})
                f = {"id": target.get("targetId"), "name": f["name"],
                     "mimeType": target.get("targetMimeType")}
            items.append(f)
        token = resp.get("nextPageToken")
        if not token:
            return items

`find_one(items, test)` trả về phần tử đầu tiên thoả điều kiện `test`, hoặc `None` nếu không có. Khi có nhiều phần tử khớp, nó in cảnh báo để bạn kiểm tra lại.

In [ ]:
def find_one(items, test):
    """Phần tử đầu tiên thoả test(item), hoặc None."""
    matches = [f for f in items if test(f)]
    if len(matches) > 1:
        names = ", ".join(f["name"] for f in matches)
        print(f"    ! {len(matches)} phần tử khớp ({names}), lấy '{matches[0]['name']}'")
    return matches[0] if matches else None

`download_file(creds, file_id, path)` tải một tệp Drive về `path`. Hàm này chạy song song nhiều luồng. Thư viện `googleapiclient` không an toàn khi nhiều luồng dùng chung một kết nối, nên mỗi luồng tự tạo kết nối Drive riêng.

In [ ]:
def download_file(creds, file_id, path):
    """Tải tệp file_id về path (mỗi luồng một kết nối Drive)."""
    if not hasattr(_tls, "drive"):
        _tls.drive = build("drive", "v3", credentials=creds, cache_discovery=False)
    request = _tls.drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with io.FileIO(path, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = downloader.next_chunk(num_retries=3)
    return path

`download_folder(drive, creds, folder_id, dst)` tải cả một thư mục Drive, kể cả thư mục con, về `dst`. Hàm làm theo hai bước:
1. Liệt kê hết các tệp.
2. Tải song song bằng `DOWNLOAD_WORKERS` luồng.

Tệp Google Docs/Sheets bị bỏ qua. Hàm trả về số tệp đã tải.

In [ ]:
def download_folder(drive, creds, folder_id, dst):
    """Tải đệ quy folder_id về dst, trả về số tệp đã tải."""
    jobs, stack = [], [(folder_id, dst)]
    while stack:
        fid, local = stack.pop()
        for f in list_children(drive, fid):
            path = os.path.join(local, f["name"])
            if is_folder(f):
                stack.append((f["id"], path))
            elif not f["mimeType"].startswith(GOOGLE_APPS_PREFIX):
                jobs.append((f["id"], path))
    os.makedirs(dst, exist_ok=True)
    with ThreadPoolExecutor(DOWNLOAD_WORKERS) as pool:
        futures = [pool.submit(download_file, creds, fid, path) for fid, path in jobs]
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc="  tải poses3d", leave=False):
            fut.result()
    return len(jobs)

`zip_and_upload(drive, src_dir, parent_id, zip_name)` nén `src_dir` rồi upload tệp zip vào thư mục Drive `parent_id`. Hàm trả về File ID của tệp zip.

Bên trong zip có sẵn thư mục gốc `poses3d/`, nên giải nén ra là được ngay `poses3d/*.npy`.

In [ ]:
def zip_and_upload(drive, src_dir, parent_id, zip_name):
    """Nén src_dir thành zip_name, upload vào parent_id, trả về file ID."""
    parent_dir, folder = os.path.split(src_dir.rstrip("/"))
    base = os.path.join(parent_dir, os.path.splitext(zip_name)[0])
    archive = shutil.make_archive(base, "zip", root_dir=parent_dir, base_dir=folder)
    media = MediaFileUpload(archive, mimetype="application/zip", resumable=True)
    created = drive.files().create(
        body={"name": zip_name, "parents": [parent_id]}, media_body=media,
        fields="id", supportsAllDrives=True,
    ).execute(num_retries=3)
    return created["id"]

## 4. GT: `poses3d.zip` (bước 2–3)
`ensure_poses3d_zip(drive, creds, processed_id, action)` trả về `z1`, tức File ID của `processed_data/poses3d.zip`.
- Nếu đã có `poses3d.zip` thì lấy ID của nó và dừng.
- Nếu chưa có thì tải `poses3d` về `LOCAL_WORK`, nén, rồi upload lên cùng cấp với `poses3d`.
- Nếu không có `poses3d`, `poses3d` rỗng, hoặc Drive báo lỗi (ví dụ không có quyền ghi) thì trả về `None`.

Thư mục tạm luôn bị xoá sau khi xong, để ổ đĩa Colab không bị đầy.

In [ ]:
def ensure_poses3d_zip(drive, creds, processed_id, action):
    """Bước 2–3: z1 = file ID của processed_data/poses3d.zip (None nếu không có)."""
    items = list_children(drive, processed_id)
    existing = find_one(items, lambda f: f["name"] == ZIP_NAME and not is_folder(f))
    if existing:
        print(f"  {ZIP_NAME} đã có: {existing['id']}")
        return existing["id"]
    poses = find_one(items, lambda f: f["name"] == "poses3d" and is_folder(f))
    if poses is None:
        print("  ! không có thư mục poses3d")
        return None
    work = os.path.join(LOCAL_WORK, action)
    shutil.rmtree(work, ignore_errors=True)
    try:
        n = download_folder(drive, creds, poses["id"], os.path.join(work, "poses3d"))
        if n == 0:
            print("  ! poses3d rỗng, không tạo zip")
            return None
        z1 = zip_and_upload(drive, os.path.join(work, "poses3d"), processed_id, ZIP_NAME)
        print(f"  đã tạo {ZIP_NAME} ({n} tệp): {z1}")
        return z1
    except HttpError as e:
        print(f"  ! lỗi Drive khi tạo {ZIP_NAME}: {e}")
        return None
    finally:
        shutil.rmtree(work, ignore_errors=True)

## 5. Tìm các sequence (bước 1)
`find_sequences(drive, test_id)` duyệt thư mục `test` theo từng tầng, sâu tối đa `MAX_DEPTH` cấp, để tìm các thư mục `processed_data`.
- Thư mục chứa `processed_data` chính là thư mục X. Tên của nó là `action`.
- Khi đã thấy `processed_data` trong X thì hàm không đi sâu thêm vào X. Nhờ vậy nó không duyệt `exo/` hay `images/`.

Hàm trả về danh sách `(X, id thư mục X, id processed_data)`, sắp theo X.

In [ ]:
def find_sequences(drive, test_id):
    """Bước 1: [(X, id thư mục X, id processed_data), ...] dưới test."""
    found, level = [], [("", test_id)]
    for _ in range(MAX_DEPTH):
        next_level = []
        for name, folder_id in level:
            children = [f for f in list_children(drive, folder_id) if is_folder(f)]
            processed = find_one(children, lambda f: f["name"] == "processed_data")
            if processed is not None:
                found.append((name, folder_id, processed["id"]))
            else:
                next_level += [(f["name"], f["id"]) for f in children]
        level = next_level
    return sorted(found)

## 6. Camera (bước 4–5)
`list_cam_folders(drive, exo_id)` lấy các thư mục `camY` trong `exo` và sắp theo số Y tăng dần.

Y được giữ dạng chuỗi như trong tên thư mục, ví dụ `"07"`. Như vậy tên tệp `harmony4D_X_cam07_...` khớp đúng.

In [ ]:
def list_cam_folders(drive, exo_id):
    """[(Y, thư mục camY), ...] trong exo, Y là chuỗi, sắp theo số."""
    cams = []
    for f in list_children(drive, exo_id):
        m = CAM_RE.match(f["name"])
        if m and is_folder(f):
            cams.append((m.group(1), f))
    return sorted(cams, key=lambda c: int(c[0]))

`find_original_video(drive, cam_items, action, cam)` tìm video gốc `harmony4D_X_camY_ffmpeg*.mp4` và trả về File ID của nó (giá trị `h4d_X_camY`).
1. Tìm trong `camY` trước.
2. Nếu không thấy, tìm tiếp trong `camY/images/`, vì notebook `render_mp4_colab.ipynb` mặc định ghi video vào đó. Bước này lọc theo tên ngay trên Drive, nên không phải liệt kê ảnh.

In [ ]:
def find_original_video(drive, cam_items, action, cam):
    """Bước 4a: file ID của harmony4D_X_camY_ffmpeg*.mp4, hoặc None."""
    stem = f"harmony4D_{action}_cam{cam}_ffmpeg"
    test = lambda f: f["name"].startswith(stem) and f["name"].lower().endswith(".mp4")
    video = find_one(cam_items, test)
    if video is None:
        images = find_one(cam_items, lambda f: f["name"] == "images" and is_folder(f))
        if images is not None:
            video = find_one(list_children(drive, images["id"], stem), test)
    return video["id"] if video else None

`find_wham_files(drive, cam_items, action, cam)` vào thư mục `harmony4D_X_camY_wham` trong `camY` và tìm hai tệp:
- tệp `.pkl` có cả X và Y trong tên → `pkl_file_ID`
- tệp `.mp4` có `output_X_camY` trong tên → `output_wham_video`

Hàm trả về `(pkl_file_ID, output_wham_video)`. Tệp nào không có thì ứng với `None`.

In [ ]:
def find_wham_files(drive, cam_items, action, cam):
    """Bước 4b + 5: (pkl_file_ID, output_wham_video), thiếu thì None."""
    wham_name = f"harmony4D_{action}_cam{cam}_wham"
    wham = find_one(cam_items, lambda f: f["name"].lower() == wham_name.lower()
                    and is_folder(f))
    if wham is None:
        return None, None
    items = [f for f in list_children(drive, wham["id"]) if not is_folder(f)]
    pkl = find_one(items, lambda f: f["name"].lower().endswith(".pkl")
                   and action in f["name"] and cam in f["name"])
    video = find_one(items, lambda f: f["name"].lower().endswith(".mp4")
                     and f"output_{action}_cam{cam}" in f["name"])
    return (pkl or {}).get("id"), (video or {}).get("id")

`process_cam(drive, action, cam, cam_folder, z1)` xử lý một thư mục `camY` và trả về một dòng kết quả dạng dict. Hàm cũng in một dòng tóm tắt: ✓ là tìm thấy, ✗ là thiếu.

In [ ]:
def process_cam(drive, action, cam, cam_folder, z1):
    """Một thư mục camY → một dòng kết quả (dict)."""
    items = list_children(drive, cam_folder["id"])
    original = find_original_video(drive, items, action, cam)
    pkl_id, wham_video = find_wham_files(drive, items, action, cam)
    mark = lambda v: "✓" if v else "✗"
    print(f"    cam{cam}: pkl {mark(pkl_id)}  video gốc {mark(original)}  "
          f"video WHAM {mark(wham_video)}")
    return {"action": action, "cam": cam, "pkl_file_ID": pkl_id,
            "original_video": original, "output_wham_video": wham_video, "z1": z1}

## 7. Một sequence
`process_action(drive, creds, action, action_id, processed_id)` xử lý một sequence X:
1. Lấy `z1` (bước 2–3).
2. Vào `exo` và xử lý từng `camY` (bước 4–5).

Hàm trả về danh sách dòng, mỗi camera một dòng.

In [ ]:
def process_action(drive, creds, action, action_id, processed_id):
    """Một sequence X → danh sách dòng kết quả (mỗi camera một dòng)."""
    print(f"\n[{action}]")
    z1 = ensure_poses3d_zip(drive, creds, processed_id, action)
    exo = find_one(list_children(drive, action_id),
                   lambda f: f["name"] == "exo" and is_folder(f))
    if exo is None:
        print("  ! không có thư mục exo")
        return []
    return [process_cam(drive, action, cam, folder, z1)
            for cam, folder in list_cam_folders(drive, exo["id"])]

## 8. Xuất kết quả (bước 6–7)
`make_json_entry(row)` tạo một phần tử của danh sách `"test"`.
- ID nào thiếu thì ghi chuỗi rỗng `""`.
- `skip = true` khi thiếu `pkl_file_ID` hoặc `z1`.

In [ ]:
def make_json_entry(row):
    """Bước 7: một phần tử của danh sách "test"."""
    return {
        "name": f"h4d_{row['action']}_cam{row['cam']}",
        "wham_pkl": row["pkl_file_ID"] or "",
        "gt": {"type": "harmony4d", "path": row["z1"] or ""},
        "output_wham_video": row["output_wham_video"] or "",
        "skip": not (row["pkl_file_ID"] and row["z1"]),
        "fps": FPS,
    }

`write_sheet(gc, rows, now)` ghi kết quả vào Google Sheet `SHEET_NAME` trong My Drive. Sheet chưa có thì được tạo mới.
- Các cột: STT, file PKL, original_video, wham_video, GT, cam, action, time.
- Chạy lại thì ghi tiếp xuống dưới, STT đánh tiếp.
- Giá trị được ghi nguyên văn (RAW), nên ID không bị đổi thành số và `07` không mất số 0.

Hàm trả về link của Sheet.

In [ ]:
def write_sheet(gc, rows, now):
    """Bước 6: ghi rows vào SHEET_NAME, trả về URL của Sheet."""
    try:
        sh = gc.open(SHEET_NAME)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)
    ws = sh.sheet1
    if not ws.row_values(1):
        ws.append_row(SHEET_HEADER, value_input_option="RAW")
    start = len(ws.col_values(1))  # số dòng đã có, kể cả dòng tiêu đề
    values = [[start + i, r["pkl_file_ID"] or "", r["original_video"] or "",
               r["output_wham_video"] or "", r["z1"] or "", r["cam"], r["action"], now]
              for i, r in enumerate(rows)]
    if values:
        ws.append_rows(values, value_input_option="RAW")
    return f"https://docs.google.com/spreadsheets/d/{sh.id}"

`save_json(drive, entries, local_path)` lưu nội dung `{"test": entries}` vào `local_path`. Sau đó nó upload tệp lên thư mục gốc My Drive. Nếu đã có tệp cùng tên thì ghi đè. Hàm trả về File ID trên Drive.

In [ ]:
def save_json(drive, entries, local_path):
    """Ghi {"test": entries} ra local_path và lên My Drive, trả về file ID."""
    with open(local_path, "w", encoding="utf-8") as fh:
        json.dump({"test": entries}, fh, ensure_ascii=False, indent=2)
    name = os.path.basename(local_path)
    media = MediaFileUpload(local_path, mimetype="application/json")
    old = drive.files().list(
        q=f"name = '{name}' and 'root' in parents and trashed = false",
        fields="files(id)",
    ).execute(num_retries=3).get("files", [])
    if old:
        req = drive.files().update(fileId=old[0]["id"], media_body=media, fields="id")
    else:
        req = drive.files().create(body={"name": name}, media_body=media, fields="id")
    return req.execute(num_retries=3)["id"]

## 9. Chạy
`main()` gọi lần lượt bước 1 đến bước 7. Nếu Drive báo lỗi ở một sequence, sequence đó bị bỏ qua và các sequence còn lại vẫn chạy. Hàm trả về `(rows, entries)`.

In [ ]:
def main():
    """Chạy bước 1–7, trả về (rows, entries)."""
    creds, drive, gc = auth_services()
    test = find_one(list_children(drive, ROOT_FOLDER_ID),
                    lambda f: f["name"] in TEST_FOLDER_NAMES and is_folder(f))
    if test is None:
        raise SystemExit(f"Không thấy thư mục {TEST_FOLDER_NAMES} trong {ROOT_FOLDER_ID}")
    sequences = find_sequences(drive, test["id"])
    print(f"Tìm thấy {len(sequences)} sequence có processed_data trong '{test['name']}'")
    rows = []
    for action, action_id, processed_id in sequences:
        try:
            rows += process_action(drive, creds, action, action_id, processed_id)
        except HttpError as e:
            print(f"  ! bỏ qua {action}: {e}")
    now = datetime.now(ZoneInfo(TIMEZONE)).strftime("%Y-%m-%d %H:%M:%S")
    entries = [make_json_entry(r) for r in rows]
    url = write_sheet(gc, rows, now)
    file_id = save_json(drive, entries, LOCAL_JSON)
    n_ok = sum(not e["skip"] for e in entries)
    print(f"\nXong: {len(rows)} camera, {n_ok} dùng được (skip = false).")
    print(f"Sheet: {url}\nJSON : {LOCAL_JSON} (Drive ID {file_id})")
    return rows, entries

Chạy toàn bộ quy trình, tải tệp JSON về máy và hiện bảng kết quả.

In [ ]:
rows, entries = main()
files.download(LOCAL_JSON)
pd.DataFrame(rows)